# 3. Policy Experiments: IM, VM Frequency, MPOR and Eligible Collateral

The policy experiments share common price paths and exogenous default dates to reduce Monte Carlo noise.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
from src.otc_margining import ModelConfig,simulate_prices,simulate_default_days,simulate_liquidity,summarize_simulation
cfg=ModelConfig(paths=20000,horizon=252,maturity=300,mpor=10,seed=42)
prices=simulate_prices(cfg,seed=100,extra_days=cfg.mpor)
default_days=simulate_default_days(cfg,seed=200)

## A. Initial-margin multiplier

In [ ]:
rows=[]
for k in [1.0,1.5,2.0,2.5]:
    result=simulate_liquidity(prices,default_days,cfg,mode='fire_sale',im_multiplier=k)
    sm=summarize_simulation(result,cfg)
    rows.append({'IM multiplier':k,'IM ($)':cfg.initial_margin(multiplier=k),'Investor default':sm['default_rate'],'Liquidity default':sm['liquidity_default_rate'],'Fire-sale cost/path':sm['mean_fire_sale_cost_per_path'],'Dealer mean loss':sm['mean_dealer_loss'],'Conditional P99 dealer loss':sm['conditional_p99']})
im_results=pd.DataFrame(rows)
im_results

## B. Weekly VM and eligible illiquid collateral

In [ ]:
weekly=simulate_liquidity(prices,default_days,cfg,mode='fire_sale',vm_frequency=5)
eligible=simulate_liquidity(prices,default_days,cfg,mode='eligible_collateral')
pd.DataFrame([summarize_simulation(fire,cfg) if 'fire' in globals() else summarize_simulation(simulate_liquidity(prices,default_days,cfg,mode='fire_sale'),cfg),summarize_simulation(weekly,cfg),summarize_simulation(eligible,cfg)],index=['Daily VM','Weekly VM','Eligible illiquid VM collateral']).T

## C. MPOR sensitivity

In [ ]:
mpor_rows=[]
for m in [10,5,3]:
    p=simulate_prices(cfg,seed=100,extra_days=m)
    result=simulate_liquidity(p,default_days,cfg,mode='fire_sale',mpor=m)
    sm=summarize_simulation(result,cfg)
    mpor_rows.append({'MPOR':m,'IM':cfg.initial_margin(mpor=m),'Investor default':sm['default_rate'],'Liquidity default':sm['liquidity_default_rate'],'Fire-sale cost/path':sm['mean_fire_sale_cost_per_path'],'Dealer mean loss':sm['mean_dealer_loss'],'Conditional P99 loss':sm['conditional_p99']})
pd.DataFrame(mpor_rows)